# Follow-up v2: step 4a on the leak_v2 GRU input

Trains six GRU models with uniform resampling (`configs/followup_v2.yaml`): the step 4a conditions with two
seeds each, and a same-compute pair (baseline and method, both trained for twice as many steps). Then scores
them on when to guess, with `gru_uniform` from followup_v1 as the baseline. Results are on validation data; the
test split is not opened. Every comparison was fixed beforehand.

**Before running:**
1. In **Input**, the `AI_pictionary` notebook output must be the **followup_v1 version** (it holds
   `runs/predictions_all/*.npz`, including `gru_uniform_s42.npz`). The notebook stops early if it cannot find them.
2. Keep your Kaggle Dataset attached and **Internet on**.
3. Choose **GPU T4 x2**, then **Save Version → Save & Run All**.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Trung4n/AI-dictionary.git"
BRANCH = "pipeline-step1"
REPO = Path("/kaggle/working/repo")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("No GITHUB_TOKEN secret: cloning without one (works for a public repository).")

url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
os.chdir("/kaggle/working")  # a re-run may still be inside the folder deleted next
if REPO.exists():
    shutil.rmtree(REPO)
done = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)], capture_output=True, text=True)
if done.returncode != 0:
    message = done.stderr.replace(token, "***") if token else done.stderr
    raise RuntimeError(f"clone failed:\n{message}")
# Keep the token out of files that Kaggle may save with the notebook output.
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
%cd /kaggle/working/repo
!pip install -q pytest
tests = subprocess.run("PYTHONPATH=src python -m pytest tests -q -k 'not processed_data'", shell=True, capture_output=True, text=True)
print(tests.stdout[-1500:])
if tests.returncode != 0:
    raise RuntimeError("tests failed: stopping before using GPU quota")

# Saved predictions of followup_v1: the folder holding the most .npz files that include gru_uniform.
candidates = [p.parent for p in Path("/kaggle/input").rglob("gru_uniform_s42.npz")]
if not candidates:
    raise RuntimeError("followup_v1 predictions not found: attach the followup_v1 version of the AI_pictionary output")
OLD_PREDICTIONS = max(candidates, key=lambda d: len(list(d.glob("*.npz"))))
names = sorted(p.name for p in OLD_PREDICTIONS.glob("*.npz"))
print("saved predictions:", OLD_PREDICTIONS, names)
if not {"gru_uniform_s42.npz", "gru_uniform_s43.npz"} <= set(names):
    raise RuntimeError("both gru_uniform seeds are needed")


In [ ]:
MAX_MINUTES = 420
TOP = Path("/kaggle/working/runs/followup_v2")
!PYTHONPATH=src python -m quickdraw_exp.experiments --matrix configs/followup_v2.yaml --max-minutes {MAX_MINUTES}
# Checkpoint predictions of the new models (GPU). The summary printed here lacks the baseline: that is expected.
!PYTHONPATH=src python -m quickdraw_exp.anytime --matrix configs/followup_v2.yaml --runs {TOP} --out {TOP}/predictions --num-workers 4


In [ ]:
# All predictions in one folder (saved ones and new ones), kept in this version's output for later steps.
ALL = Path("/kaggle/working/runs/predictions_all")
ALL.mkdir(parents=True, exist_ok=True)
for folder in (OLD_PREDICTIONS, TOP / "predictions"):
    for p in folder.glob("*.npz"):
        target = ALL / p.name
        if not target.exists():
            shutil.copy(p, target)
print(sorted(p.name for p in ALL.glob("*.npz")))
!PYTHONPATH=src python -m quickdraw_exp.anytime --matrix configs/followup_v2.yaml --predictions {ALL} --out /kaggle/working/runs/followup_v2_anytime


In [ ]:
import json

training = json.loads((TOP / "results.json").read_text())
print("training", "COMPLETE" if training["complete"] else "INCOMPLETE, missing: " + ", ".join(training["missing"]))
v1 = json.loads(Path("reports/followup_v1/results.json").read_text())["table"]
print("\nTop-1 by ink fraction, val 1000 per class (gru_uniform from reports/followup_v1):")
for name, row in [("gru_uniform", v1["gru_uniform"]), *training["table"].items()]:
    print(f"  {name:28s} seeds {row['seeds']}", {f: round(v["top1"]["mean"], 4) for f, v in row["by_frac"].items()},
          "early", round(row["early_top1"]["mean"], 4))

result = json.loads(Path("/kaggle/working/runs/followup_v2_anytime/results.json").read_text())
print("\nmissing conditions:", result["missing_conditions"])
print("\nPre-registered comparisons (AUC, evaluation half):")
for row in result["comparisons"]:
    if row["difference"] is None:
        print(f"  {row['name']}: not available")
    else:
        print(f"  {row['name']}: {row['a_mean']:.4f} vs {row['b_mean']:.4f}, difference {row['difference']:+.4f}  "
              f"(a {[round(v, 4) for v in row['a_values']]}, b {[round(v, 4) for v in row['b_values']]})")
print("\nRisk control and AUC per condition:")
for condition, policies in result["table"].items():
    for policy, e in policies.items():
        if "earliness_at_gap_0.02" in e:
            print(f"  {condition:28s} {policy:7s} seeds {e['seeds']} auc {e['auc']['mean']:.4f}  earliness@gap5% "
                  f"{e['earliness_at_gap_0.05']['mean']:.3f} met {e['met_gap_0.05']}")
for run, ref in result["reference"].items():
    print(f"  {run}: end of drawing {ref['end_of_drawing']['accuracy']:.4f}")
for name in training["missing"]:
    log = TOP / f"{name}.log"
    if log.exists():
        print(f"--- {name} ---")
        print("\n".join(log.read_text().splitlines()[-15:]))
